In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
CHECKPOINT_PATH = os.path.join(SPARKYLLM, "checkpoints", "gpt_medium_phase2.pth")
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")
META_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "train_long_meta.json")

Mounted at /content/drive


In [3]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ==================== MODEL DEFINITION ====================
BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64
FF_HIDDEN_DIM = 4 * EMBED_DIM

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList([TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.0) for _ in range(NUM_LAYERS)])
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks: x = block(x)
        return self.lm_head(self.final_norm(x))

# ==================== LOAD MODEL ====================
with open(META_PATH, "r") as f:
    meta = json.load(f)
VOCAB_SIZE = int(meta["vocab_size"])
if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleGPT(VOCAB_SIZE).to(device)

# Load checkpoint with diagnostics
if not os.path.exists(CHECKPOINT_PATH):
    print(f"CHECKPOINT NOT FOUND: {CHECKPOINT_PATH}")
    print("Listing available checkpoints:")
    ckpt_dir = os.path.dirname(CHECKPOINT_PATH)
    if os.path.isdir(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            print(f"  {f}")
    else:
        print(f"  Directory does not exist: {ckpt_dir}")
else:
    file_size = os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024
    print(f"Checkpoint: {CHECKPOINT_PATH} ({file_size:.1f} MB)")

    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    print(f"Checkpoint keys: {len(state_dict)}")

    # Show a few raw keys to understand the format
    raw_keys = list(state_dict.keys())[:5]
    print(f"Sample keys: {raw_keys}")

    # Clean _orig_mod. prefix from torch.compile
    clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

    # Match against model
    model_state = model.state_dict()
    matched = {}
    mismatched_shape = []
    missing_in_ckpt = []
    extra_in_ckpt = []

    for k, v in clean_state.items():
        if k in model_state:
            if v.shape == model_state[k].shape:
                matched[k] = v
            else:
                mismatched_shape.append((k, v.shape, model_state[k].shape))
        else:
            extra_in_ckpt.append(k)

    for k in model_state:
        if k not in clean_state:
            missing_in_ckpt.append(k)

    print(f"\nMatched:          {len(matched)} / {len(model_state)} model keys")
    if mismatched_shape:
        print(f"Shape mismatch:   {len(mismatched_shape)}")
        for k, ckpt_shape, model_shape in mismatched_shape:
            print(f"  {k}: checkpoint {ckpt_shape} vs model {model_shape}")
    if missing_in_ckpt:
        print(f"Missing in ckpt:  {len(missing_in_ckpt)}")
        for k in missing_in_ckpt[:10]:
            print(f"  {k}")
    if extra_in_ckpt:
        print(f"Extra in ckpt:    {len(extra_in_ckpt)}")
        for k in extra_in_ckpt[:10]:
            print(f"  {k}")

    if len(matched) == 0:
        print("\nERROR: No keys matched! The checkpoint is incompatible with this model.")
    else:
        model.load_state_dict(matched, strict=False)
        print(f"\nLoaded {len(matched)} layers successfully.")

model.eval()

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
eot_id = tokenizer.token_to_id("<|endoftext|>")

print(f"\nModel on {device} | Vocab: {VOCAB_SIZE} | Params: {sum(p.numel() for p in model.parameters()):,}")

Checkpoint: /content/drive/MyDrive/sparkyllm/checkpoints/gpt_medium_phase2.pth (2562.1 MB)
Checkpoint keys: 293
Sample keys: ['_orig_mod.token_embedding.weight', '_orig_mod.position_embedding.weight', '_orig_mod.blocks.0.norm1.weight', '_orig_mod.blocks.0.norm1.bias', '_orig_mod.blocks.0.attn.c_attn.weight']

Matched:          293 / 293 model keys

Loaded 293 layers successfully.

Model on cuda | Vocab: 32000 | Params: 671,613,440


In [4]:
@torch.no_grad()
def generate(prompt, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2):
    ids = tokenizer.encode(prompt).ids
    tokens = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_tokens):
        context = tokens[:, -BLOCK_SIZE:]
        logits = model(context)[:, -1, :]

        # Repetition penalty (HF-style)
        if repetition_penalty > 1.0:
            seen = set(tokens[0].tolist())
            for tid in seen:
                if tid < logits.size(-1):
                    if logits[0, tid] > 0:
                        logits[0, tid] /= repetition_penalty
                    else:
                        logits[0, tid] *= repetition_penalty

        # Temperature
        logits = logits / temperature

        # Top-k
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # Top-p (nucleus)
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = float('-inf')
            logits = torch.zeros_like(logits).scatter(1, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        if next_token.item() == eot_id:
            break

        tokens = torch.cat([tokens, next_token], dim=1)

    output_ids = tokens[0].tolist()
    return tokenizer.decode(output_ids)

In [5]:
# ==================== TEST 1: PERPLEXITY ====================
# Measures how well the model predicts held-out text. Lower = better.
# Rough guide: <50 decent, <20 good, <10 very good for small LMs.
import numpy as np
import math

EVAL_FILE = os.path.join(SPARKYLLM, "datasets_pretrain", "training_data_long.txt")
EVAL_CHUNKS = 50
CHUNK_LENGTH = BLOCK_SIZE

@torch.no_grad()
def compute_perplexity(text_path, num_chunks, chunk_len):
    # Stream-tokenize line by line to avoid loading everything into memory
    all_ids = []
    target_tokens = (num_chunks + 1) * (chunk_len + 1)

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ids = tokenizer.encode(line).ids
            all_ids.extend(ids)
            if len(all_ids) >= target_tokens:
                break

    total_tokens = len(all_ids)
    print(f"Tokenized {total_tokens:,} tokens for evaluation")

    if total_tokens < chunk_len + 1:
        print("Not enough tokens to evaluate!")
        return None

    losses = []
    rng = np.random.default_rng(42)
    starts = rng.integers(0, total_tokens - chunk_len - 1, size=num_chunks)

    for s in starts:
        chunk = all_ids[s : s + chunk_len + 1]
        x = torch.tensor([chunk[:-1]], dtype=torch.long, device=device)
        y = torch.tensor([chunk[1:]], dtype=torch.long, device=device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())

    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss)

    print(f"\nPerplexity Results ({num_chunks} chunks of {chunk_len} tokens):")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Perplexity:   {ppl:.2f}")
    print(f"  Min Loss:     {min(losses):.4f} (best chunk)")
    print(f"  Max Loss:     {max(losses):.4f} (worst chunk)")
    return ppl

perplexity = compute_perplexity(EVAL_FILE, EVAL_CHUNKS, CHUNK_LENGTH)

Tokenized 39,246 tokens for evaluation

Perplexity Results (50 chunks of 768 tokens):
  Average Loss: 2.7822
  Perplexity:   16.16
  Min Loss:     1.9865 (best chunk)
  Max Loss:     3.2107 (worst chunk)


In [6]:
# ==================== TEST 2: REPETITION RATE ====================
# Measures how often the model repeats itself. Lower = better.
# Checks 2-gram, 3-gram, and 4-gram repetition rates.

from collections import Counter

def repetition_rate(text, n):
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(ngrams) if ngrams else 0.0

test_prompts = [
    # Literary
    "Once upon a time",
    "The knight drew his sword and",
    # Web / informational (C4-style)
    "The best way to learn programming is",
    "Scientists recently discovered that",
    "According to a new study published in",
    # Conversational
    "I think the most important thing about",
    "Here are three reasons why",
]

print("Repetition Rate Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'2-gram':>8} {'3-gram':>8} {'4-gram':>8}")
print("-" * 70)

all_outputs = []
for prompt in test_prompts:
    output = generate(prompt, max_tokens=300, temperature=0.8, top_k=40)
    all_outputs.append(output)
    r2 = repetition_rate(output, 2)
    r3 = repetition_rate(output, 3)
    r4 = repetition_rate(output, 4)
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {r2:>7.1%} {r3:>7.1%} {r4:>7.1%}")

# Overall
combined = " ".join(all_outputs)
print("-" * 70)
print(f"{'OVERALL':<40} {repetition_rate(combined, 2):>7.1%} {repetition_rate(combined, 3):>7.1%} {repetition_rate(combined, 4):>7.1%}")
print("\nGuide: <10% is good, 10-25% is okay, >25% is concerning")

Repetition Rate Analysis
Prompt                                     2-gram   3-gram   4-gram
----------------------------------------------------------------------
Once upon a time                            3.9%    0.0%    0.0%
The knight drew his sword and               1.2%    0.4%    0.0%
The best way to learn programming is        0.7%    0.0%    0.0%
Scientists recently discovered that         0.0%    0.0%    0.0%
According to a new study published in       0.0%    0.0%    0.0%
I think the most important thing abou...    0.5%    0.0%    0.0%
Here are three reasons why                  3.5%    0.8%    0.0%
----------------------------------------------------------------------
OVERALL                                     5.5%    0.5%    0.0%

Guide: <10% is good, 10-25% is okay, >25% is concerning


In [7]:
# ==================== TEST 3: TOKEN DIVERSITY ====================
# Measures unique tokens / total tokens. Higher = better.
# Low diversity suggests mode collapse (model stuck in a rut).

def token_diversity(text):
    ids = tokenizer.encode(text).ids
    total = len(ids)
    unique = len(set(ids))
    return unique, total, unique / total if total > 0 else 0.0

print("Token Diversity Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'Unique':>7} {'Total':>7} {'Ratio':>8}")
print("-" * 70)

total_unique_all = set()
total_count_all = 0

for prompt, output in zip(test_prompts, all_outputs):
    unique, total, ratio = token_diversity(output)
    ids = tokenizer.encode(output).ids
    total_unique_all.update(ids)
    total_count_all += total
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {unique:>7} {total:>7} {ratio:>7.1%}")

print("-" * 70)
overall_ratio = len(total_unique_all) / total_count_all if total_count_all > 0 else 0
print(f"{'OVERALL':<40} {len(total_unique_all):>7} {total_count_all:>7} {overall_ratio:>7.1%}")
print(f"\nVocab utilization: {len(total_unique_all)} / {VOCAB_SIZE} tokens used ({len(total_unique_all)/VOCAB_SIZE:.1%})")
print("Guide: >30% diversity is good, <15% is concerning")

Token Diversity Analysis
Prompt                                    Unique   Total    Ratio
----------------------------------------------------------------------
Once upon a time                             206     303   68.0%
The knight drew his sword and                215     306   70.3%
The best way to learn programming is         117     148   79.1%
Scientists recently discovered that           73      85   85.9%
According to a new study published in         66      79   83.5%
I think the most important thing abou...     191     275   69.5%
Here are three reasons why                   210     297   70.7%
----------------------------------------------------------------------
OVERALL                                      649    1493   43.5%

Vocab utilization: 649 / 32000 tokens used (2.0%)
Guide: >30% diversity is good, <15% is concerning


In [8]:
# ==================== TEST 4: COHERENCE CHECK ====================
# Generates multiple continuations from the same prompt.
# If the model is coherent, outputs should be varied but thematically consistent.

coherence_prompts = [
    "The knight drew his sword and",
    "Deep in the mountains there lived",
    "The key to a successful business is",
    "New research suggests that climate change",
]

NUM_SAMPLES = 3

for prompt in coherence_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    for i in range(NUM_SAMPLES):
        output = generate(prompt, max_tokens=150, temperature=0.9, top_k=50)
        continuation = output[len(tokenizer.decode(tokenizer.encode(prompt).ids)):]
        print(f"\n--- Sample {i+1} ---")
        print(continuation.strip()[:500])


PROMPT: The knight drew his sword and

--- Sample 1 ---
threw it in the hands of a woman, who was standing at the feet."I'm not going to go with you," said she; "and as I came down into town on time.""That's my opinion?" answered Natásha, when she spoke about her mother: "Well, what is that?"Alyosha laughed suddenly after he had heard of him, and went to tell Alexey how many years since then--for three or four months he has been married! She knew that some day he would marry me for an hour with her father; but he did not know where it took him now fr

--- Sample 2 ---
put it in the fire. He is wearing his shield, which has a red-grey background with yellow spots. She was at last attacked by an owl. The dragon went down to look for her.

--- Sample 3 ---
said unto him, It is better not to be punished for your crime. He came out of the castle in the morning and took her oath from you as a child; so that it may be kept by fire or by night.10:21 Then he began to cry all kinds of mischief.

In [9]:
# ==================== TEST 5: LOSS ON KNOWN vs RANDOM TEXT ====================
# Compares model loss on in-domain text (from training data) vs out-of-domain text.
# Since we now train on books + C4 web text, out-of-domain needs to be
# things genuinely absent: non-English, highly technical, or structured formats.

import random

@torch.no_grad()
def compute_loss_on_text(text):
    ids = tokenizer.encode(text).ids
    if len(ids) < 2:
        return None
    ids = ids[:BLOCK_SIZE + 1]
    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([ids[1:]], dtype=torch.long, device=device)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
    return loss.item()

# Load passages from training data (stream to avoid OOM)
EVAL_FILE = os.path.join(SPARKYLLM, "datasets_pretrain", "training_data_long.txt")
rng = random.Random(42)
in_domain_passages = []
with open(EVAL_FILE, "r", encoding="utf-8", errors="ignore") as f:
    full_text = f.read(500_000)

for _ in range(5):
    start = rng.randint(0, max(0, len(full_text) - 5000))
    passage = full_text[start:start + 3000].strip()
    if passage:
        in_domain_passages.append(passage)

# Out-of-domain: things NOT in C4 English or classic books
out_of_domain_passages = [
    # Non-English
    ("Japanese",
     "量子コンピューティングの分野では、トポロジカル量子ビットを用いた"
     "誤り訂正技術の開発が進んでいます。超伝導回路において、ミリケルビン"
     "温度で動作する表面符号アーキテクチャの最近の進歩により、論理エラー率を"
     "符号距離の増加に伴って指数関数的に抑制できることが実証されています。"),

    # Structured data / markup
    ("JSON/API",
     '{"users": [{"id": 1, "name": "Alice", "roles": ["admin", "editor"], '
     '"created_at": "2024-01-15T08:30:00Z", "metadata": {"login_count": 342, '
     '"last_ip": "192.168.1.100"}}, {"id": 2, "name": "Bob", "roles": ["viewer"], '
     '"created_at": "2024-03-22T14:15:00Z", "metadata": {"login_count": 17, '
     '"last_ip": "10.0.0.55"}}], "pagination": {"page": 1, "total": 1523}}'),

    # Mathematical notation
    ("Math/LaTeX",
     "Consider the Riemann zeta function zeta(s) = sum_{n=1}^{infty} n^{-s} for "
     "Re(s) > 1. By analytic continuation, zeta(s) extends to a meromorphic function "
     "on the entire complex plane with a simple pole at s=1. The functional equation "
     "zeta(s) = 2^s pi^{s-1} sin(pi s/2) Gamma(1-s) zeta(1-s) relates values at s "
     "and 1-s. The Riemann Hypothesis asserts that all non-trivial zeros lie on Re(s)=1/2."),

    # Assembly / machine code
    ("Assembly",
     "section .text\nglobal _start\n_start:\n    mov rax, 1\n    mov rdi, 1\n"
     "    lea rsi, [msg]\n    mov rdx, 13\n    syscall\n    mov rax, 60\n"
     "    xor rdi, rdi\n    syscall\nsection .data\n"
     "msg: db 'Hello World', 0xa, 0"),

    # Chemical formulas / medical
    ("Chemistry",
     "The synthesis of acetylsalicylic acid (C9H8O4) proceeds via acetylation of "
     "salicylic acid (2-hydroxybenzoic acid, C7H6O3) with acetic anhydride "
     "((CH3CO)2O) in the presence of phosphoric acid catalyst at 85C for 15 min. "
     "Yield: 78.3%. mp 135-136C. IR (KBr): 1753 cm-1 (C=O ester), 1689 cm-1 "
     "(C=O acid). 1H NMR (CDCl3): d 2.36 (s, 3H, COCH3), 7.12-8.13 (m, 4H, ArH)."),
]

print("Loss Comparison: In-Domain vs Out-of-Domain")
print("=" * 60)

print("\nIn-Domain (training data passages):")
in_losses = []
for i, passage in enumerate(in_domain_passages):
    loss = compute_loss_on_text(passage)
    if loss is not None:
        in_losses.append(loss)
        preview = passage[:60].replace('\n', ' ') + "..."
        print(f"  [{i+1}] Loss: {loss:.4f}  |  {preview}")

print("\nOut-of-Domain (unseen text):")
out_losses = []
for i, (domain, passage) in enumerate(out_of_domain_passages):
    loss = compute_loss_on_text(passage)
    if loss is not None:
        out_losses.append(loss)
        print(f"  [{i+1}] Loss: {loss:.4f}  |  [{domain}]")

avg_in = sum(in_losses) / len(in_losses) if in_losses else 0
avg_out = sum(out_losses) / len(out_losses) if out_losses else 0
gap = avg_out - avg_in

print(f"\n{'='*60}")
print(f"  Avg In-Domain Loss:      {avg_in:.4f}  (PPL: {math.exp(avg_in):.2f})")
print(f"  Avg Out-of-Domain Loss:  {avg_out:.4f}  (PPL: {math.exp(avg_out):.2f})")
print(f"  Gap:                     {gap:.4f}")
print(f"\nVerdict: ", end="")
if gap > 1.0:
    print("GOOD - model clearly learned in-domain patterns")
elif gap > 0.3:
    print("OK - model shows some domain specialization")
elif gap > 0:
    print("WEAK - model barely distinguishes in-domain from random text")
else:
    print("BAD - model performs worse on its own training data")

Loss Comparison: In-Domain vs Out-of-Domain

In-Domain (training data passages):
  [1] Loss: 3.2339  |  lt put in the breastplate of judgment the Urim and the Thumm...
  [2] Loss: 3.4272  |  This shall not be thine heir; but he that shall come forth o...
  [3] Loss: 2.8440  |  he called Night. And the evening and the morning were the fi...
  [4] Loss: 3.0621  |  o the rings of the ephod with a lace of blue, that it might ...
  [5] Loss: 3.2569  |  longeth for your daughter: I pray you give her him to wife. ...

Out-of-Domain (unseen text):
  [1] Loss: 3.0929  |  [Japanese]
  [2] Loss: 3.0884  |  [JSON/API]
  [3] Loss: 4.3409  |  [Math/LaTeX]
  [4] Loss: 6.3016  |  [Assembly]
  [5] Loss: 4.0810  |  [Chemistry]

  Avg In-Domain Loss:      3.1648  (PPL: 23.68)
  Avg Out-of-Domain Loss:  4.1810  (PPL: 65.43)
  Gap:                     1.0161

Verdict: GOOD - model clearly learned in-domain patterns


In [10]:
# ==================== INTERACTIVE CHAT ====================
# Type text, the model continues it. Context accumulates across turns.
# Type "reset" to clear context, "quit" to stop.

@torch.no_grad()
def generate_streaming(token_ids, max_tokens=150, temperature=0.8, top_k=40, repetition_penalty=1.2):
    """Generate from a list of token IDs, print as it goes, return updated list."""
    tokens = torch.tensor([token_ids], dtype=torch.long, device=device)
    new_ids = []

    for _ in range(max_tokens):
        context = tokens[:, -BLOCK_SIZE:]
        logits = model(context)[:, -1, :]

        # Repetition penalty
        if repetition_penalty > 1.0:
            seen = set(tokens[0].tolist())
            for tid in seen:
                if tid < logits.size(-1):
                    if logits[0, tid] > 0:
                        logits[0, tid] /= repetition_penalty
                    else:
                        logits[0, tid] *= repetition_penalty

        logits = logits / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tid = next_token.item()

        if tid == eot_id:
            print("<|end|>", end="")
            break

        new_ids.append(tid)
        tokens = torch.cat([tokens, next_token], dim=1)
        piece = tokenizer.decode([tid])
        print(piece, end="", flush=True)

    print()
    return token_ids + new_ids

# --- Main loop ---
context_ids = []
print("=" * 60)
print("INTERACTIVE MODE")
print("  Your text gets added to context, model continues from there.")
print("  Commands: 'reset' = clear context, 'quit' = exit")
print("=" * 60)

while True:
    try:
        user_input = input("\nYou > ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nExiting.")
        break

    if user_input.lower() == "quit":
        break
    if user_input.lower() == "reset":
        context_ids = []
        print("Context cleared.")
        continue
    if not user_input:
        continue

    # Tokenize user input and append to context
    new_tokens = tokenizer.encode(user_input).ids
    context_ids.extend(new_tokens)

    # Trim context if too long (keep last BLOCK_SIZE tokens)
    if len(context_ids) > BLOCK_SIZE:
        context_ids = context_ids[-BLOCK_SIZE:]

    print(f"\nModel ({len(context_ids)} tokens in context) > ", end="", flush=True)
    context_ids = generate_streaming(context_ids, max_tokens=150, temperature=0.8, top_k=40)
    print(f"  [{len(context_ids)} tokens total]")

INTERACTIVE MODE
  Your text gets added to context, model continues from there.
  Commands: 'reset' = clear context, 'quit' = exit

Exiting.
